# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a complete guide for loading and exploring the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and is accessible via:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print('Dataset Title:')
print(metadata.name)
print('\nDataset Description:\n')
print(metadata.description)

# Optionally view full top-level metadata
# from pprint import pprint
# pprint(metadata.to_json())

## 2. Data Overview
Review available record sets and fields using their `@id` identifiers. This step helps understand dataset structure.

In [ ]:
# List all record sets in the dataset via metadata
record_set_ids = []
for rs in metadata.recordSet:
    record_set_ids.append(rs['@id'])
print('Available record set @ids:')
for rs_id in record_set_ids:
    print(rs_id)

# Show all fields (columns) for each record set
for rs in metadata.recordSet:
    print(f"\nRecord Set @id: {rs['@id']}")
    if 'field' in rs:
        print('Fields:')
        for field in rs['field']:
            print(f"  Field @id: {field['@id']}  Name: {field.get('name', 'N/A')}")
    else:
        print('No fields in this record set.')

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. Reference record set and field `@id` values from above overview.

In [ ]:
# Prepare a list of record set @ids
record_sets = record_set_ids
dataframes = {}

for record_set_id in record_sets:
    print(f"\nLoading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(2))

# Example: choose the first record set for downstream demonstration
chosen_record_set_id = record_sets[0] if record_sets else None
if chosen_record_set_id:
    print(f"\nWorking with record set @id: {chosen_record_set_id}")
    print(f"Columns: {dataframes[chosen_record_set_id].columns.tolist()}")
    dataframes[chosen_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping by key attributes. All fields are referenced by their `@id` values.

In [ ]:
# For demonstration, use the first record set
record_set_id = chosen_record_set_id
df = dataframes.get(record_set_id)

# Find a numeric field to analyze by checking its dtype
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id:
    print(f"Numeric field selected for analysis: {numeric_field_id}")

    # Filter records for numeric_field > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    mean_val = filtered_df[numeric_field_id].mean()
    std_val = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val
    print(f"Normalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Find a potential group field (categorical, not numeric)
    group_field_id = None
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            group_field_id = col
            break

    if group_field_id and group_field_id in filtered_df:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field available for EDA in this record set.")

## 5. Visualization
Visualize the distribution of the numeric field and its relationship with the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f'Distribution of {numeric_field_id} in {record_set_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} grouped by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No numeric field found for visualization.')

## 6. Conclusion
This notebook demonstrated loading, exploring, and processing the FAIR² dataset package using `mlcroissant`. We reviewed dataset structure by referencing all entities by their `@id`, loaded tabular records into DataFrames, performed simple EDA, and visualized example distributions. This provides a foundation for downstream clinical and statistical analyses on second primary colorectal cancer data in cancer survivors.